# Create Searle Scholars Awards (FELLOWSHIP PATTERN, WordPress + WP REST)

Searle Scholars Program is a US biomedical/chemistry research-funder for
early-career faculty (~15 scholars/yr since 1981, $300K over 3 years).
Funded through the Searle Funds at The Chicago Community Trust;
administered by Kinship Foundation.

Source is the foundation's own site (searlescholars.org). The
`/current-scholars/` grid renders ~6 recent cohorts as category-tagged
`<article class="obb-content ... category-{year}-scholar">` cards with
name + profile URL + institution + research title in well-structured
HTML. Latest-cohort announcement post(s) not yet folded into the grid
are picked up via the WP REST `/wp-json/wp/v2/posts` endpoint and parsed
from their `<p class="wp-block-paragraph">` body markup.

**Awarding body:** Searle Scholars Program - F4320314849 (US, DOI 10.13039/100012316)

**Schema choices:**
- One row per (year, name) pair. `funder_award_id` = `searle-{year}-{name_slug}`.
- `funder_scheme` = `Searle Scholars Program` (single program, single scheme).
- `funding_type` = `research`.
- `amount` / `currency` = USD 300,000 per scholar (official, verified from the 2025 class-announcement post: "Each receives an award of $300,000"). 100% coverage.
- `start_year` = announcement year; `end_year` = `year + 2` (3-year term per official announcement).
- `lead_investigator.affiliation.country` = `US` — Searle eligibility per the program's own page is US universities and research centers only.
- No `declined` column.

**Coverage scope:** ~105 scholars from the 7 most recent cohorts (2020-2026 at time of ingest). The pre-2020 archive (~615 scholars 1981-2019) is NOT exposed on the current site — the foundation publishes the aggregate "707 total since 1981" figure but not the historical list. Documented as a Step 0 follow-up in the tracker.

**Source:** https://searlescholars.org/current-scholars/ + https://searlescholars.org/wp-json/wp/v2/posts

**S3 location:** `s3a://openalex-ingest/awards/searle_scholars/searle_scholars.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.searle_scholars_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/searle_scholars/searle_scholars.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.searle_scholars_raw;

## Step 1.5: Inspect raw + per-year/institution distribution

Expected: ~105 rows, 7 cohorts (2020-2026), 15 scholars/yr, all USD 300K, 100% on name + institution + research_title + profile_url.

In [ ]:
%sql
DESCRIBE openalex.awards.searle_scholars_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.searle_scholars_raw LIMIT 5;

In [ ]:
%sql
-- Cohort sanity. Each year should have 15 scholars (Searle's standing cohort size).
SELECT year, COUNT(*) AS rows,
       COUNT(institution) AS has_inst,
       COUNT(research_title) AS has_title,
       COUNT(profile_url) AS has_url
FROM openalex.awards.searle_scholars_raw
GROUP BY year
ORDER BY year;

In [ ]:
%sql
-- Top institutions (sanity check we didn't merge fields).
SELECT institution, COUNT(*) AS rows
FROM openalex.awards.searle_scholars_raw
WHERE institution IS NOT NULL
GROUP BY institution
ORDER BY rows DESC
LIMIT 10;

## Step 1.6: Fail-fast - verify Searle Scholars Program funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320314849;

## Step 2: Transform to award schema

Per-row `display_name` = `Searle Scholar - {Name} ({year})`. `description` is the recipient's research project title. `start_year` = announcement year, `end_year` = announcement year + 2 (3-year award term).

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.searle_scholars_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320314849  -- Searle Scholars Program
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Searle Scholar - ', n.name, ' (', n.year, ')') AS display_name,
    n.research_title AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'research' AS funding_type,
    'Searle Scholars Program' AS funder_scheme,
    'searle_scholars' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.year AS INT) AS start_year,
    TRY_CAST(n.year AS INT) + 2 AS end_year,  -- 3-year term per official announcement
    CASE
        WHEN n.name IS NULL OR n.name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.institution AS name,
                CAST('US' AS STRING) AS country,  -- Searle eligibility is US-only per the program page
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    COALESCE(n.profile_url, n.landing_page_url) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.searle_scholars_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 133

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'searle_scholars' AND priority = 133;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    133 AS priority  -- Searle Scholars Program
FROM openalex.awards.searle_scholars_awards;

## Step 6: Verification

Expect ~105 rows, 100% amount (all USD 300,000), 100% name + institution + research_title + profile_url + start_year + end_year, year range 2020-2026.

In [ ]:
%sql
SELECT COUNT(*) AS total_searle_rows FROM openalex.awards.searle_scholars_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.searle_scholars_awards;

In [ ]:
%sql
-- §6.3 Data completeness. All percentages should be 100.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.searle_scholars_awards;

In [ ]:
%sql
-- §6.7 amount check. Expect 100% USD 300,000 across all rows.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    MIN(amount) AS min_amount, MAX(amount) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.searle_scholars_awards;

In [ ]:
%sql
-- Year/cohort sanity. Each cohort should be ~15 scholars.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.searle_scholars_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
-- Funder struct sanity - all rows must point to Searle Scholars Program.
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.searle_scholars_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Sample 10 rows for eyeball QA.
SELECT id, SUBSTRING(display_name, 1, 60) AS title,
       funder_scheme AS program, funding_type, start_year, end_year, amount,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       SUBSTRING(lead_investigator.affiliation.name, 1, 45) AS institution
FROM openalex.awards.searle_scholars_awards
ORDER BY start_year DESC, family
LIMIT 10;